In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
DEBUG_FLAG:bool = True

spark = SparkSession.builder.appName("IoT_Temp_Analytics").getOrCreate()

1. Load sensor data

In [0]:
# Step 1: Create the raw sensor data (simulates incoming stream records)

"""
Hour, ID, Temperature
---------------------
8, R3, NULL
8, R3,70
8, R1, 79
9, R1, 71
9, R5, NULL
9, R4, 72
9, R2, NULL
10, R3, 72
10, R3, NULL
10, R2, 75
11, R2, 78
10, R1, NULL
10, R5, 78
12, R2, 79
12, R3, NULL
12, R5, 80
12, R1, 70
13, R4, 77
13, R3, 72
13, R2, NULL
13, R5, 78
8, R1, 80
"""

schema = StructType([
    StructField(name="hour", dataType=IntegerType(), nullable=False),
    StructField(name="sensor_id", dataType=StringType(), nullable=False),
    StructField(name="temperature", dataType=IntegerType(), nullable=True)  # NULL-capable
])

raw_data = [
    (8,  "R3", None),
    (8,  "R3", 70.0),
    (8,  "R1", 79.0),
    (9,  "R1", 71.0),
    (9,  "R5", None),
    (9,  "R4", 72.0),
    (9,  "R2", None),
    (10, "R3", 72.0),
    (10, "R3", None),
    (10, "R2", 75.0),
    (11, "R2", 78.0),
    (10, "R1", None),
    (10, "R5", 78.0),
    (12, "R2", 79.0),
    (12, "R3", None),
    (12, "R5", 80.0),
    (12, "R1", 70.0),
    (13, "R4", 77.0),
    (13, "R3", 72.0),
    (13, "R2", None),
    (13, "R5", 78.0),
    (8,  "R1", 80.0),   # out-of-order duplicate
    # (7,  None, 100),    # should not load as mandatory sensor_id is null
]

raw_df = spark.createDataFrame(raw_data, schema=schema)

if DEBUG_FLAG:
    raw_df.show(50)

2. Dedupe

In [0]:
# Step 2: Deduplicate — for same hour+sensor, keep only the max temperature
# NULLs are ignored by max() naturally in Spark
deduped_df = (
    raw_df
    .groupBy("hour", "sensor_id")
    .agg(F.max("temperature").alias("temperature"))  # NULLs ignored by max()
)

if DEBUG_FLAG:
    deduped_df.orderBy("hour", "sensor_id").show()

3. Window

In [0]:
# Step 3: Forward-fill within each sensor, ordered by hour

sensor_window = (
    Window
    .partitionBy("sensor_id")
    .orderBy("hour")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

# Last non-null value up to current row
filled_df = deduped_df.withColumn(
    "filled_temp",
    F.last("temperature", ignorenulls=True).over(sensor_window)
)

if DEBUG_FLAG:
    filled_df.orderBy("hour", "sensor_id").show()

4. Intermediate dataframe to explain the temperature ffill status

In [0]:
# Step 4. Explain logic using status column
# A sensor is only "active" after its FIRST non-null value
# Rows before first non-null → filled_temp will still be null → mark as INACTIVE

COLUMN__RESOLVED_TEMP:str = "resolved_temperature"
COLUMN__STATUS:str = "status"

unfiltered_final_df = (
    filled_df
    .withColumn(
        COLUMN__RESOLVED_TEMP,
        F.when(F.col("filled_temp").isNull(), None)   # not yet active
         .otherwise(F.col("filled_temp"))
    )
    .withColumn(
        COLUMN__STATUS,
        F.when(F.col("filled_temp").isNull(), F.lit("INACTIVE"))
         .when(F.col("temperature").isNull(), F.lit("FORWARD_FILLED"))
         .otherwise(F.lit("ACTUAL"))
    )
    .select("hour", "sensor_id", COLUMN__RESOLVED_TEMP, COLUMN__STATUS)
)

if DEBUG_FLAG:
    unfiltered_final_df.orderBy("hour", "sensor_id").show()

5. Filter inactive sensor records 

In [0]:
# Step 5. Only include rows where sensor is active
final_df = unfiltered_final_df.filter(F.col("resolved_temperature").isNotNull())

final_df.orderBy("hour", "sensor_id").show(100)

ALTERNATE APPROACH: Using SQL for Delta Live Table

In [0]:
db_sql = """
WITH 
Hourly_Aggregated AS (
  -- Handle duplicate readings by grabbing the MAX temperature per hour per sensor
  SELECT 
    hour, 
    sensor_id, 
    MAX(temperature) AS temperature
  FROM {sensor_df}
  GROUP BY hour, sensor_id
)
,
Windowed_Table AS (
  SELECT 
    hour, 
    sensor_id, 
    -- Forward fill NULLs considering previous valid values
    LAST_VALUE(temperature) IGNORE NULLS OVER (
      PARTITION BY sensor_id 
      ORDER BY hour 
      ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS Adjusted_Temperature
  FROM Hourly_Aggregated
)
SELECT
  *
FROM Windowed_Table
WHERE Adjusted_Temperature IS NOT NULL
ORDER BY hour, sensor_id;

"""

"""
Expected result: 
|hour|sensor_id|resolved_temperature|        status|
+----+---------+--------------------+--------------+
|   8|       R1|                  80|        ACTUAL|
|   8|       R3|                  70|        ACTUAL|
|   9|       R1|                  71|        ACTUAL|
|   9|       R4|                  72|        ACTUAL|
|  10|       R1|                  71|FORWARD_FILLED|
|  10|       R2|                  75|        ACTUAL|
|  10|       R3|                  72|        ACTUAL|
|  10|       R5|                  78|        ACTUAL|
|  11|       R2|                  78|        ACTUAL|
|  12|       R1|                  70|        ACTUAL|
|  12|       R2|                  79|        ACTUAL|
|  12|       R3|                  72|FORWARD_FILLED|
|  12|       R5|                  80|        ACTUAL|
|  13|       R2|                  79|FORWARD_FILLED|
|  13|       R3|                  72|        ACTUAL|
|  13|       R4|                  77|        ACTUAL|
|  13|       R5|                  78|        ACTUAL|
+----+---------+--------------------+--------------+
"""

dlt_df = spark.sql(db_sql, sensor_df=raw_df)
dlt_df.show(100)